# Continual Learning Analysis

This notebook is dedicated to post-hoc analysis. Training runs should only emit artifacts;
this notebook discovers them and computes metrics without re-running training.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any, Dict, Iterable, Optional

import numpy as np
import pandas as pd

In [ ]:
def _matches_config(args: Dict[str, Any], target_config: Optional[Dict[str, Any]]) -> bool:
    if not target_config:
        return True
    for key, expected in target_config.items():
        if args.get(key) != expected:
            return False
    return True


def fetch_latest_experiment(
    target_config: Optional[Dict[str, Any]] = None,
    root_dirs: Iterable[str] = ("logs", "outputs"),
) -> Dict[str, Any]:
    candidates = []
    for root in root_dirs:
        root_path = Path(root)
        if not root_path.exists():
            continue
        for args_path in root_path.glob("**/args.json"):
            run_dir = args_path.parent
            try:
                args_payload = json.loads(args_path.read_text())
            except Exception:
                continue
            if not _matches_config(args_payload, target_config):
                continue
            tasks_path = run_dir / "results_tasks.json"
            if not tasks_path.exists():
                continue
            candidates.append((args_path.stat().st_mtime, run_dir, args_payload))

    if not candidates:
        raise FileNotFoundError("No matching experiments found.")

    candidates.sort(key=lambda item: item[0], reverse=True)
    _, run_dir, args_payload = candidates[0]
    results_tasks = json.loads((run_dir / "results_tasks.json").read_text())
    step_path = run_dir / "results_step_eval.json"
    if step_path.exists():
        results_step_eval = json.loads(step_path.read_text())
    else:
        results_step_eval = []

    return {
        "run_dir": str(run_dir),
        "args": args_payload,
        "results_tasks": results_tasks,
        "results_step_eval": results_step_eval,
    }

In [ ]:
# Example:
# exp = fetch_latest_experiment({"patch_mode": "normal", "codebook_mode": "ema_mean"})
# exp["run_dir"]
# len(exp["results_tasks"])


## Task-Eval vs Step-Eval
Task-eval comes from `results_tasks.json` and reflects linear probing after each task stage. Step-eval comes from `results_step_eval.json` and reflects in-training logits sampled at fixed intervals. Keep them separate when analyzing dynamics.

In [ ]:
def build_task_eval_matrix(results_tasks: list[dict[str, object]]) -> np.ndarray:
    rows = []
    for stage in results_tasks:
        row = []
        for metric in stage.get("task_eval", []):
            value = metric.get("accuracy")
            row.append(np.nan if value is None else float(value))
        rows.append(row)
    if not rows:
        return np.empty((0, 0), dtype=float)
    return np.array(rows, dtype=float)


def average_accuracy(task_matrix: np.ndarray) -> float:
    if task_matrix.size == 0:
        return float("nan")
    return float(np.nanmean(task_matrix[-1]))


def backward_transfer(task_matrix: np.ndarray) -> float:
    if task_matrix.size == 0:
        return float("nan")
    num_tasks = min(task_matrix.shape[0], task_matrix.shape[1])
    diag = np.diag(task_matrix[:num_tasks, :num_tasks])
    final = task_matrix[-1, :num_tasks]
    if num_tasks <= 1:
        return float("nan")
    return float(np.nanmean(final[:-1] - diag[:-1]))

In [ ]:
# Example (kept commented to avoid accidental execution errors):
# exp = fetch_latest_experiment({"patch_mode": "normal", "codebook_mode": "ema_mean"})
# task_matrix = build_task_eval_matrix(exp["results_tasks"])
# aa = average_accuracy(task_matrix)
# bwt = backward_transfer(task_matrix)

#
# step_eval_df = pd.DataFrame(exp["results_step_eval"])  # Step-eval only
# task_eval_df = pd.DataFrame(exp["results_tasks"])     # Task-eval only
